# Pilot: do commit summaries help find the right fix commit?

Many commits have a bare message, so a bug report written in English matches only their diff.
Before summarising all 4,883 commits, this pilot embeds about 1,100 of them twice on a GPU:
once as indexed today, once with a short plain-English `Summary:` line (written by the 27B model
from the commit's own message and diff, never from a bug report). It then ranks the 141 bug
reports' gold fix commits both ways.

**Before you start (on your laptop)**
1. `uv run python eval/commit_summary_pilot.py prepare --random 1000`
2. `uv run python eval/commit_summary_pilot.py summarize --url <27B server> --token <token>`
   (about an hour; resumable)
3. `uv run python eval/commit_summary_pilot.py package` writes
   `eval/commit_summary_pilot/pilot_input.zip`

**Here**
1. Runtime → Change runtime type → a **GPU** (an A100 is fastest; a T4 works but is slow).
2. Run all. When asked, upload `pilot_input.zip`.
3. The last cell prints a first answer (dense search only) and downloads `pilot_vectors.npz`.

**Back on your laptop:** `uv run python eval/commit_summary_pilot.py score --vectors
path/to/pilot_vectors.npz` adds the hybrid (dense + BM25) comparison the agent really uses.

Ranks here are within about 1,100 commits, not the full 4,883: they show direction, not the
final score. A gain still has to be confirmed through the agent.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!command -v uv >/dev/null || pip install -q uv
# Pinned to the versions in the project's uv.lock. Colab already ships a CUDA build of torch.
!uv pip install --system -q "chromadb==1.5.9" "fastapi==0.162.0" "gitpython==3.2.1" "langchain-chroma==1.1.0" "langchain-core==1.6.13" "langchain-text-splitters==1.1.3" "langgraph==1.2.19" "pydantic==2.14.1" "pydantic-settings==2.16.1" "rank-bm25==0.2.2" "rich==15.0.0" "sentence-transformers==5.7.0" "transformers==4.57.6" "tree-sitter==0.26.0" "tree-sitter-language-pack==1.22.1" "typer==0.27.3" "uvicorn==0.54.0"

In [ ]:
# The repo_rag package, embedded by notebooks/build_colab_indexer.py (content hash 802157f52695).
# Don't edit this cell: change src/repo_rag and regenerate the notebook.
import base64, io, pathlib, zipfile

ARCHIVE_B64 = "..."

SRC = pathlib.Path("/content/repo_rag_src")
with zipfile.ZipFile(io.BytesIO(base64.b64decode(ARCHIVE_B64))) as archive:
    archive.extractall(SRC)
print(f"unpacked {len(list(SRC.rglob('*.py')))} files into {SRC} (content hash 802157f52695)")

In [ ]:
# eval/commit_summary_pilot.py, embedded by notebooks/build_commit_summary_pilot.py
# (content hash 947cd13bb9b6). Don't edit this cell: edit the script and regenerate the notebook.
import base64, pathlib

pathlib.Path("/content/commit_summary_pilot.py").write_bytes(base64.b64decode("IiIiUGlsb3Q6IGRvIHBsYWluLUVuZ2xpc2ggY29tbWl0IHN1bW1hcmllcyBtYWtlIHRoZSByaWdodCBjb21taXQgZWFzaWVyIHRvIGZpbmQ/CgpNYW55IHBzZi9yZXF1ZXN0cyBjb21taXRzIGhhdmUgYSBiYXJlIG1lc3NhZ2UgKCJGaXhlZCAjMTc0IiwgaXNzdWUgcmVmZXJlbmNlcyBhcmUgc3RyaXBwZWQgZnJvbQp0aGUgaW5kZXgpLCBzbyBhIGJ1ZyByZXBvcnQgd3JpdHRlbiBpbiBFbmdsaXNoIG1hdGNoZXMgdGhlaXIgZGlmZiBwb29ybHkuIFRoaXMgcGlsb3QgYWRkcyBhIHNob3J0CnN1bW1hcnkgd3JpdHRlbiBieSB0aGUgYW5zd2VyIG1vZGVsIGZyb20gdGhlIGNvbW1pdCdzIG93biBtZXNzYWdlIGFuZCBkaWZmLCBhbmQgY2hlY2tzIHdoZXRoZXIgdGhlCmdvbGQgZml4IGNvbW1pdCByYW5rcyBoaWdoZXIuIEl0IGlzIGEgc21hbGwsIGNoZWFwIHRlc3QgcnVuIGJlZm9yZSBzdW1tYXJpc2luZyBhbGwgNCw4ODMgY29tbWl0cy4KClN0ZXBzIChsYXB0b3AgdW5sZXNzIG5vdGVkKToKICAgIDEuIHByZXBhcmUgICAgcGljayB0aGUgMTQxLXBhaXIgZ29sZCBjb21taXRzIHBsdXMgTiByYW5kb20gb3RoZXJzIGZyb20gdGhlIGxvY2FsIGNvbW1pdCBpbmRleAogICAgICAgICAgICAgICAgICBhbmQgd3JpdGUgZXZhbC9jb21taXRfc3VtbWFyeV9waWxvdC97Y29tbWl0cyxxdWVyaWVzfS5qc29ubAogICAgMi4gc3VtbWFyaXplICBhc2sgdGhlIHJ1bm5pbmcgMjdCIHNlcnZlciBmb3Igb25lIHN1bW1hcnkgcGVyIHBpbG90IGNvbW1pdCAocmVzdW1hYmxlOyB0aGUgc2VydmVyCiAgICAgICAgICAgICAgICAgIGhhbmRsZXMgb25lIHJlcXVlc3QgYXQgYSB0aW1lLCBzbyB+NCBzIGVhY2gpCiAgICAzLiBwYWNrYWdlICAgIHppcCBjb21taXRzICsgc3VtbWFyaWVzICsgcXVlcmllcyBmb3IgdGhlIENvbGFiIG5vdGVib29rCiAgICA0LiAoQ29sYWIpICAgIG5vdGVib29rcy9jb21taXRfc3VtbWFyeV9waWxvdF9vbl9jb2xhYi5pcHluYiBlbWJlZHMgYm90aCB2ZXJzaW9ucyBvbiBhIEdQVSBhbmQKICAgICAgICAgICAgICAgICAgcmV0dXJucyBwaWxvdF92ZWN0b3JzLm5wegogICAgNS4gc2NvcmUgICAgICByYW5rIHRoZSBnb2xkIGNvbW1pdCB3aXRoIGFuZCB3aXRob3V0IHRoZSBzdW1tYXJ5IGluc2lkZSB0aGUgcGlsb3QgY29ycHVzCgpMZWFrYWdlIHJ1bGU6IHRoZSBzdW1tYXJpc2VyIHNlZXMgb25seSB0aGUgY29tbWl0J3Mgb3duIGluZGV4ZWQgdGV4dCAobWVzc2FnZSBhbmQgZGlmZiwgaXNzdWUKcmVmZXJlbmNlcyBhbHJlYWR5IHN0cmlwcGVkKS4gSXQgbmV2ZXIgc2VlcyBhIGJ1ZyByZXBvcnQsIGFuIGlzc3VlIG9yIGEgUFIuIEdvbGQgYW5kIHJhbmRvbSBjb21taXRzCmdldCBleGFjdGx5IHRoZSBzYW1lIHRyZWF0bWVudCwgc28gdGhlIHN1bW1hcmllcyBjYW5ub3QgZmF2b3VyIHRoZSBhbnN3ZXJzLgoKUmFua3MgYXJlIHdpdGhpbiBhIH4xLDEwMC1jb21taXQgY29ycHVzLCBub3QgdGhlIGZ1bGwgNCw4ODMsIHNvIHRoZXkgc2hvdyBkaXJlY3Rpb24sIG5vdCB0aGUgZmluYWwKc2NvcmUuIEFueSBnYWluIG11c3QgYmUgcmUtbWVhc3VyZWQgdGhyb3VnaCB0aGUgYWdlbnQgYmVmb3JlIGl0IGNvdW50cy4KCiAgICB1diBydW4gcHl0aG9uIGV2YWwvY29tbWl0X3N1bW1hcnlfcGlsb3QucHkgcHJlcGFyZSAtLXJhbmRvbSAxMDAwCiAgICB1diBydW4gcHl0aG9uIGV2YWwvY29tbWl0X3N1bW1hcnlfcGlsb3QucHkgc3VtbWFyaXplIC0tdXJsIFUgLS10b2tlbiBUCiAgICB1diBydW4gcHl0aG9uIGV2YWwvY29tbWl0X3N1bW1hcnlfcGlsb3QucHkgcGFja2FnZQogICAgdXYgcnVuIHB5dGhvbiBldmFsL2NvbW1pdF9zdW1tYXJ5X3BpbG90LnB5IHNjb3JlIC0tdmVjdG9ycyBwYXRoL3RvL3BpbG90X3ZlY3RvcnMubnB6CiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBhc3QKaW1wb3J0IGpzb24KaW1wb3J0IHJhbmRvbQppbXBvcnQgcmUKaW1wb3J0IHN5cwppbXBvcnQgdGltZQppbXBvcnQgemlwZmlsZQpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgU2VxdWVuY2UKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUk9PVCAvICJzcmMiKSkKClJFUE8gPSAicHNmL3JlcXVlc3RzIgpQQUlSUyA9IFJPT1QgLyAiZXZhbC9kYXRhc2V0cy9waGFzZTVfYnVnZml4X3YyL3JlcG9zL3BzZl9fcmVxdWVzdHMvcGFpcnMuanNvbmwiCldPUksgPSBST09UIC8gImV2YWwvY29tbWl0X3N1bW1hcnlfcGlsb3QiCkNPTU1JVFMgPSBXT1JLIC8gImNvbW1pdHMuanNvbmwiClFVRVJJRVMgPSBXT1JLIC8gInF1ZXJpZXMuanNvbmwiClNVTU1BUklFUyA9IFdPUksgLyAic3VtbWFyaWVzLmpzb25sIgpQQUNLQUdFID0gV09SSyAvICJwaWxvdF9pbnB1dC56aXAiClJFU1VMVCA9IFdPUksgLyAicmVzdWx0Lmpzb24iCgpCVUdfQ0hBUlMgPSAxMjAwICAjIHNhbWUgY3V0IGFzIGV2YWwvYWdlbnRfZXZhbC5weSBhbmQgZXZhbC9jb21taXRfcmV0cmlldmFsX2NoZWNrLnB5ClNVTU1BUllfSU5QVVRfQ0hBUlMgPSA2MDAwICAjIHRoZSBjb21taXQgdGV4dCBzaG93biB0byB0aGUgc3VtbWFyaXNlciAoYWJvdXQgMiwwMDAgdG9rZW5zKQpTVU1NQVJZX01BWF9DSEFSUyA9IDcwMCAgIyBhIHN1bW1hcnkgaXMgY3V0IHRvIHRoaXMgYmVmb3JlIGl0IGlzIGFkZGVkIHRvIHRoZSBpbmRleGVkIHRleHQKU0VFRCA9IDIwMjYxMTE4CgpTWVNURU0gPSAoCiAgICAiWW91IHdyaXRlIHNob3J0LCBwbGFpbi1FbmdsaXNoIGRlc2NyaXB0aW9ucyBvZiBnaXQgY29tbWl0cyBmb3IgYSBzZWFyY2ggaW5kZXguICIKICAgICJVc2Ugb25seSB0aGUgY29tbWl0IHNob3duLiBOZXZlciBpbnZlbnQgZGV0YWlscyB0aGF0IGFyZSBub3QgaW4gaXQuIgopCklOU1RSVUNUSU9OID0gKAogICAgIkRlc2NyaWJlIHRoaXMgY29tbWl0IGluIHR3byBvciB0aHJlZSBwbGFpbiBzZW50ZW5jZXMsIGF0IG1vc3QgNjAgd29yZHMuIFNheSB3aGF0IGJlaGF2aW91ciAiCiAgICAib3IgcHJvYmxlbSBpdCBjaGFuZ2VzICh0aGUgc3ltcHRvbSBhIHVzZXIgd291bGQgbm90aWNlLCBpZiBpdCBmaXhlcyBhIGJ1ZyksIGFuZCBuYW1lIHRoZSAiCiAgICAibWFpbiBmdW5jdGlvbiwgY2xhc3Mgb3IgbW9kdWxlLiBEbyBub3QgcXVvdGUgY29kZSBvciBkaWZmIGxpbmVzIGFuZCBkbyBub3QgbWVudGlvbiBpc3N1ZSAiCiAgICAibnVtYmVycy4gUmVwbHkgd2l0aCB0aGUgZGVzY3JpcHRpb24gb25seS5cblxuQ09NTUlUXG4iCikKCgojIC0tLSBwdXJlIGhlbHBlcnMgKHRlc3RlZCBvZmZsaW5lKSAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgoKZGVmIGdvbGRfcHJlZml4ZXMocGFpcjogZGljdCkgLT4gbGlzdFtzdHJdOgogICAgZ29sZCA9IHBhaXJbImdvbGQiXQogICAgc2hhcyA9IGFzdC5saXRlcmFsX2V2YWwoZ29sZCkgaWYgaXNpbnN0YW5jZShnb2xkLCBzdHIpIGVsc2UgZ29sZAogICAgcmV0dXJuIFtzWzoxMl0gZm9yIHMgaW4gc2hhc10KCgpkZWYgc2VsZWN0X3BpbG90KAogICAgZG9jczogU2VxdWVuY2VbZGljdF0sIGdvbGRzOiBzZXRbc3RyXSwgbl9yYW5kb206IGludCwgc2VlZDogaW50ID0gU0VFRAopIC0+IGxpc3RbZGljdF06CiAgICAiIiJFdmVyeSBnb2xkIGNvbW1pdCBwcmVzZW50IGluIGBkb2NzYCwgcGx1cyBgbl9yYW5kb21gIG90aGVycyBkcmF3biB3aXRoIGEgZml4ZWQgc2VlZC4KCiAgICBFYWNoIGRvYyBpcyB7InNoYSIsICJ0ZXh0In0uIFRoZSBvdXRwdXQgaXMgc29ydGVkIGJ5IHNoYSBzbyB0aGUgb3JkZXIgbmV2ZXIgZGVwZW5kcyBvbiBob3cgdGhlCiAgICBzdG9yZSBoYXBwZW5lZCB0byBsaXN0IHRoZW0uIiIiCiAgICBnb2xkX2RvY3MgPSBbZCBmb3IgZCBpbiBkb2NzIGlmIGRbInNoYSJdWzoxMl0gaW4gZ29sZHNdCiAgICBvdGhlcnMgPSBzb3J0ZWQoKGQgZm9yIGQgaW4gZG9jcyBpZiBkWyJzaGEiXVs6MTJdIG5vdCBpbiBnb2xkcyksIGtleT1sYW1iZGEgZDogZFsic2hhIl0pCiAgICBwaWNrZWQgPSByYW5kb20uUmFuZG9tKHNlZWQpLnNhbXBsZShvdGhlcnMsIG1pbihuX3JhbmRvbSwgbGVuKG90aGVycykpKQogICAgcmV0dXJuIHNvcnRlZChnb2xkX2RvY3MgKyBwaWNrZWQsIGtleT1sYW1iZGEgZDogZFsic2hhIl0pCgoKZGVmIHN1bW1hcnlfbWVzc2FnZXModGV4dDogc3RyKSAtPiBsaXN0W2RpY3Rbc3RyLCBzdHJdXToKICAgIHJldHVybiBbCiAgICAgICAgeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50IjogU1lTVEVNfSwKICAgICAgICB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogSU5TVFJVQ1RJT04gKyB0ZXh0WzpTVU1NQVJZX0lOUFVUX0NIQVJTXX0sCiAgICBdCgoKX1RISU5LID0gcmUuY29tcGlsZShyIjx0aGluaz4uKj88L3RoaW5rPiIsIHJlLkRPVEFMTCkKCgpkZWYgY2xlYW5fc3VtbWFyeShyYXc6IHN0cikgLT4gc3RyOgogICAgIiIiT25lIGxpbmUsIG5vIHRoaW5raW5nIGJsb2NrLCBubyBsZWFkaW5nIGxhYmVsLCBjdXQgdG8gU1VNTUFSWV9NQVhfQ0hBUlMuIiIiCiAgICB0ZXh0ID0gX1RISU5LLnN1YigiIiwgcmF3KS5zdHJpcCgpCiAgICB0ZXh0ID0gcmUuc3ViKHIiXihzdW1tYXJ5fGRlc2NyaXB0aW9uKVxzKjpccyoiLCAiIiwgdGV4dCwgZmxhZ3M9cmUuSUdOT1JFQ0FTRSkKICAgIHJldHVybiAiICIuam9pbih0ZXh0LnNwbGl0KCkpWzpTVU1NQVJZX01BWF9DSEFSU10KCgpkZWYgd2l0aF9zdW1tYXJ5KHRleHQ6IHN0ciwgc3VtbWFyeTogc3RyKSAtPiBzdHI6CiAgICAiIiJUaGUgaW5kZXhlZCBjb21taXQgdGV4dCB3aXRoIGEgYFN1bW1hcnk6YCBsaW5lIGJlZm9yZSBgRmlsZXM6YCAoYWZ0ZXIgdGhlIG1lc3NhZ2UpLgoKICAgIEl0IGdvZXMgYmVmb3JlIHRoZSBkaWZmIG9uIHB1cnBvc2U6IHRoZSBlbWJlZGRlciB0cnVuY2F0ZXMgbG9uZyBpbnB1dHMgZnJvbSB0aGUgZW5kLCBzbyBhCiAgICBzdW1tYXJ5IHBsYWNlZCBmaXJzdCBpcyBuZXZlciBjdXQgb2ZmLiIiIgogICAgaWYgbm90IHN1bW1hcnk6CiAgICAgICAgcmV0dXJuIHRleHQKICAgIGxpbmUgPSBmIlN1bW1hcnk6IHtzdW1tYXJ5fSIKICAgIGlmICJcbkZpbGVzOiAiIGluIHRleHQ6CiAgICAgICAgcmV0dXJuIHRleHQucmVwbGFjZSgiXG5GaWxlczogIiwgZiJcbntsaW5lfVxuRmlsZXM6ICIsIDEpCiAgICBoZWFkLCBzZXAsIHJlc3QgPSB0ZXh0LnBhcnRpdGlvbigiXG4iKQogICAgcmV0dXJuIGYie2hlYWR9XG57bGluZX17c2VwfXtyZXN0fSIgaWYgc2VwIGVsc2UgZiJ7aGVhZH1cbntsaW5lfSIKCgpkZWYgcmFua3NfZm9yKG9yZGVyOiBTZXF1ZW5jZVtpbnRdLCBzaGFzOiBTZXF1ZW5jZVtzdHJdLCBnb2xkczogU2VxdWVuY2Vbc3RyXSkgLT4gaW50IHwgTm9uZToKICAgICIiIjEtYmFzZWQgcmFuayBvZiB0aGUgZmlyc3QgZ29sZCBjb21taXQgaW4gYSBiZXN0LWZpcnN0IGxpc3Qgb2YgZG9jdW1lbnQgaW5kZXhlcy4iIiIKICAgIHdhbnRlZCA9IHNldChnb2xkcykKICAgIGZvciBwb3NpdGlvbiwgaW5kZXggaW4gZW51bWVyYXRlKG9yZGVyLCBzdGFydD0xKToKICAgICAgICBpZiBzaGFzW2luZGV4XVs6MTJdIGluIHdhbnRlZDoKICAgICAgICAgICAgcmV0dXJuIHBvc2l0aW9uCiAgICByZXR1cm4gTm9uZQoKCmRlZiBzdW1tYXJpemVfcmFua3MocmFua3M6IFNlcXVlbmNlW2ludCB8IE5vbmVdKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIG4gPSBsZW4ocmFua3MpCgogICAgZGVmIGhpdChrOiBpbnQpIC0+IGludDoKICAgICAgICByZXR1cm4gc3VtKDEgZm9yIHIgaW4gcmFua3MgaWYgciBpcyBub3QgTm9uZSBhbmQgciA8PSBrKQoKICAgIG1yciA9IHN1bSgxIC8gciBmb3IgciBpbiByYW5rcyBpZiByIGlzIG5vdCBOb25lKSAvIG4gaWYgbiBlbHNlIDAuMAogICAgcmV0dXJuIHsKICAgICAgICAibiI6IG4sCiAgICAgICAgImhpdEAxIjogaGl0KDEpLAogICAgICAgICJoaXRANSI6IGhpdCg1KSwKICAgICAgICAiaGl0QDEwIjogaGl0KDEwKSwKICAgICAgICAiaGl0QDIwIjogaGl0KDIwKSwKICAgICAgICAibWlzc2luZyI6IHN1bSgxIGZvciByIGluIHJhbmtzIGlmIHIgaXMgTm9uZSksCiAgICAgICAgIm1yciI6IHJvdW5kKG1yciwgMyksCiAgICB9CgoKZGVmIGNvbXBhcmVfcmFua3MoYmVmb3JlOiBTZXF1ZW5jZVtpbnQgfCBOb25lXSwgYWZ0ZXI6IFNlcXVlbmNlW2ludCB8IE5vbmVdKSAtPiBkaWN0W3N0ciwgaW50XToKICAgICIiIlBhaXJ3aXNlOiBob3cgbWFueSBnb2xkIGNvbW1pdHMgbW92ZWQgdXAsIGRvd24gb3Igc3RheWVkIChOb25lIGNvdW50cyBhcyB3b3JzdCkuIiIiCiAgICB3b3JzdCA9IDEwKio5CiAgICB1cCA9IGRvd24gPSBzYW1lID0gMAogICAgZm9yIGIsIGEgaW4gemlwKGJlZm9yZSwgYWZ0ZXIsIHN0cmljdD1UcnVlKToKICAgICAgICBiLCBhID0gYiBvciB3b3JzdCwgYSBvciB3b3JzdAogICAgICAgIGlmIGEgPCBiOgogICAgICAgICAgICB1cCArPSAxCiAgICAgICAgZWxpZiBhID4gYjoKICAgICAgICAgICAgZG93biArPSAxCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2FtZSArPSAxCiAgICByZXR1cm4geyJiZXR0ZXIiOiB1cCwgIndvcnNlIjogZG93biwgInNhbWUiOiBzYW1lfQoKCmRlZiBkZW5zZV9vcmRlcihkb2NfdmVjdG9yczogbnAubmRhcnJheSwgcXVlcnlfdmVjdG9yOiBucC5uZGFycmF5KSAtPiBsaXN0W2ludF06CiAgICAiIiJEb2N1bWVudCBpbmRleGVzIGJ5IGNvc2luZSBzaW1pbGFyaXR5ICh2ZWN0b3JzIGFyZSB1bml0IGxlbmd0aCksIGJlc3QgZmlyc3QuIiIiCiAgICByZXR1cm4gbGlzdChucC5hcmdzb3J0KC0oZG9jX3ZlY3RvcnMgQCBxdWVyeV92ZWN0b3IpLCBraW5kPSJzdGFibGUiKSkKCgpkZWYgaHlicmlkX29yZGVyKAogICAgZGVuc2U6IFNlcXVlbmNlW2ludF0sIHNoYXM6IFNlcXVlbmNlW3N0cl0sIGJtMjVfaWRzOiBTZXF1ZW5jZVtzdHJdLCB3ZWlnaHRzOiBkaWN0LCBycmZfazogaW50CikgLT4gbGlzdFtpbnRdOgogICAgIiIiV2VpZ2h0ZWQgUlJGIG9mIHRoZSBkZW5zZSByYW5raW5nIGFuZCBhIEJNMjUgcmFua2luZywgdGhlIGZ1c2lvbiB0aGUgYWdlbnQgdXNlcy4iIiIKICAgIGZyb20gcmVwb19yYWcucmV0cmlldmFsLnJldHJpZXZlcnMgaW1wb3J0IHdlaWdodGVkX3JyZgoKICAgIGluZGV4X29mID0ge3NoYTogaSBmb3IgaSwgc2hhIGluIGVudW1lcmF0ZShzaGFzKX0KICAgIGZ1c2VkID0gd2VpZ2h0ZWRfcnJmKAogICAgICAgIHsiZGVuc2UiOiBbc2hhc1tpXSBmb3IgaSBpbiBkZW5zZV0sICJibTI1IjogbGlzdChibTI1X2lkcyl9LCB3ZWlnaHRzLCBycmZfawogICAgKQogICAgb3JkZXJlZCA9IFtpbmRleF9vZltzaGFdIGZvciBzaGEsIF8gaW4gZnVzZWRdCiAgICBzZWVuID0gc2V0KG9yZGVyZWQpCiAgICByZXR1cm4gb3JkZXJlZCArIFtpIGZvciBpIGluIGRlbnNlIGlmIGkgbm90IGluIHNlZW5dCgoKIyAtLS0gY29tbWFuZHMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpkZWYgcmVhZF9qc29ubChwYXRoOiBQYXRoKSAtPiBsaXN0W2RpY3RdOgogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIFtdCiAgICByZXR1cm4gW2pzb24ubG9hZHMoeCkgZm9yIHggaW4gcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpIGlmIHguc3RyaXAoKV0KCgpkZWYgd3JpdGVfanNvbmwocGF0aDogUGF0aCwgcm93czogU2VxdWVuY2VbZGljdF0pIC0+IE5vbmU6CiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwYXRoLndyaXRlX3RleHQoIlxuIi5qb2luKGpzb24uZHVtcHMocikgZm9yIHIgaW4gcm93cykgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQoKCmRlZiBwYWlycygpIC0+IGxpc3RbZGljdF06CiAgICByZXR1cm4gW2pzb24ubG9hZHMoeCkgZm9yIHggaW4gUEFJUlMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKV0KCgpkZWYgcHJlcGFyZShuX3JhbmRvbTogaW50KSAtPiBOb25lOgogICAgZnJvbSByZXBvX3JhZy5jb25maWcgaW1wb3J0IFNldHRpbmdzCiAgICBmcm9tIHJlcG9fcmFnLnN0b3JlLmVtYmVkZGluZ3MgaW1wb3J0IGdldF9lbWJlZGRpbmdzCiAgICBmcm9tIHJlcG9fcmFnLnN0b3JlLnZlY3RvcnN0b3JlIGltcG9ydCBWZWN0b3JTdG9yZQoKICAgIHNldHRpbmdzID0gU2V0dGluZ3MoKQogICAgc3RvcmUgPSBWZWN0b3JTdG9yZShzZXR0aW5ncywgZ2V0X2VtYmVkZGluZ3Moc2V0dGluZ3MpKQogICAgZG9jcyA9IFsKICAgICAgICB7InNoYSI6IHN0cihkLm1ldGFkYXRhWyJjb21taXRfc2hhIl0pLCAidGV4dCI6IGQucGFnZV9jb250ZW50fQogICAgICAgIGZvciBkIGluIHN0b3JlLnJlcG9fZG9jdW1lbnRzKCJjb21taXRzIiwgUkVQTykKICAgIF0KICAgIGFsbF9wYWlycyA9IHBhaXJzKCkKICAgIGdvbGRzID0ge2cgZm9yIHAgaW4gYWxsX3BhaXJzIGZvciBnIGluIGdvbGRfcHJlZml4ZXMocCl9CiAgICBjaG9zZW4gPSBzZWxlY3RfcGlsb3QoZG9jcywgZ29sZHMsIG5fcmFuZG9tKQogICAgcHJlc2VudCA9IHtkWyJzaGEiXVs6MTJdIGZvciBkIGluIGNob3Nlbn0KICAgIHVzYWJsZSA9IFtwIGZvciBwIGluIGFsbF9wYWlycyBpZiBhbnkoZyBpbiBwcmVzZW50IGZvciBnIGluIGdvbGRfcHJlZml4ZXMocCkpXQogICAgd3JpdGVfanNvbmwoQ09NTUlUUywgY2hvc2VuKQogICAgd3JpdGVfanNvbmwoCiAgICAgICAgUVVFUklFUywKICAgICAgICBbCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJpZCI6IHBbImlkIl0sCiAgICAgICAgICAgICAgICAic3BsaXQiOiBwWyJzcGxpdCJdLAogICAgICAgICAgICAgICAgInF1ZXJ5IjogcFsicXVlcnkiXVs6QlVHX0NIQVJTXSwKICAgICAgICAgICAgICAgICJnb2xkIjogW2cgZm9yIGcgaW4gZ29sZF9wcmVmaXhlcyhwKSBpZiBnIGluIHByZXNlbnRdLAogICAgICAgICAgICB9CiAgICAgICAgICAgIGZvciBwIGluIHVzYWJsZQogICAgICAgIF0sCiAgICApCiAgICBwcmludCgKICAgICAgICBmImNvbW1pdCBpbmRleDoge2xlbihkb2NzKX0gY29tbWl0czsgZ29sZCBzaGFzIHdhbnRlZCB7bGVuKGdvbGRzKX0sICIKICAgICAgICBmImZvdW5kIHtsZW4oZ29sZHMgJiBwcmVzZW50KX1cbnBpbG90IGNvcnB1czoge2xlbihjaG9zZW4pfSBjb21taXRzOyAiCiAgICAgICAgZiJwYWlycyB1c2FibGUge2xlbih1c2FibGUpfSBvZiB7bGVuKGFsbF9wYWlycyl9XG53cm90ZSB7Q09NTUlUU30gYW5kIHtRVUVSSUVTfSIKICAgICkKCgpkZWYgc3VtbWFyaXplKHVybDogc3RyLCB0b2tlbjogc3RyLCBsaW1pdDogaW50IHwgTm9uZSkgLT4gTm9uZToKICAgIGZyb20gcmVwb19yYWcuYWdlbnQucmVtb3RlX2xsbSBpbXBvcnQgUmVtb3RlQ2hhdE1vZGVsCgogICAgbW9kZWwgPSBSZW1vdGVDaGF0TW9kZWwodXJsLCB0b2tlbiwgdGltZW91dD0zMDAuMCkKICAgIHByaW50KCJzZXJ2ZXI6IiwgbW9kZWwuaGVhbHRoKCkpCiAgICBjb21taXRzID0gcmVhZF9qc29ubChDT01NSVRTKQogICAgaWYgbm90IGNvbW1pdHM6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgicnVuIGBwcmVwYXJlYCBmaXJzdCIpCiAgICBkb25lID0ge3JbInNoYSJdIGZvciByIGluIHJlYWRfanNvbmwoU1VNTUFSSUVTKSBpZiByLmdldCgic3VtbWFyeSIpfQogICAgdG9kbyA9IFtjIGZvciBjIGluIGNvbW1pdHMgaWYgY1sic2hhIl0gbm90IGluIGRvbmVdWzpsaW1pdF0KICAgIHByaW50KGYie2xlbihkb25lKX0gZG9uZSwge2xlbih0b2RvKX0gdG8gZG8iKQogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGZvciBudW1iZXIsIGNvbW1pdCBpbiBlbnVtZXJhdGUodG9kbywgc3RhcnQ9MSk6CiAgICAgICAgcm93OiBkaWN0W3N0ciwgQW55XSA9IHsic2hhIjogY29tbWl0WyJzaGEiXX0KICAgICAgICBmb3IgYXR0ZW1wdCBpbiAoMSwgMik6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGdlbmVyYXRpb24gPSBtb2RlbC5nZW5lcmF0ZShzdW1tYXJ5X21lc3NhZ2VzKGNvbW1pdFsidGV4dCJdKSkKICAgICAgICAgICAgICAgIHJvd1sic3VtbWFyeSJdID0gY2xlYW5fc3VtbWFyeShnZW5lcmF0aW9uLnRleHQpCiAgICAgICAgICAgICAgICByb3dbInNlY29uZHMiXSA9IHJvdW5kKGdlbmVyYXRpb24uc2Vjb25kcywgMikKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBhIGJ1ZGdldCBvciBzZXJ2ZXIgZXJyb3I6IHJldHJ5IG9uY2UsIHRoZW4gcmVjb3JkIGl0CiAgICAgICAgICAgICAgICByb3dbImVycm9yIl0gPSBmInt0eXBlKGV4YykuX19uYW1lX199OiB7c3RyKGV4YylbOjEyMF19IgogICAgICAgICAgICAgICAgdGltZS5zbGVlcCgyICogYXR0ZW1wdCkKICAgICAgICB3aXRoIFNVTU1BUklFUy5vcGVuKCJhIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgICAgICBoYW5kbGUud3JpdGUoanNvbi5kdW1wcyhyb3cpICsgIlxuIikKICAgICAgICBpZiBudW1iZXIgJSAyNSA9PSAwIG9yIG51bWJlciA9PSBsZW4odG9kbyk6CiAgICAgICAgICAgIHJhdGUgPSAodGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQpIC8gbnVtYmVyCiAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgZiJbe251bWJlcn0ve2xlbih0b2RvKX1dIHtyYXRlOi4xZn0gcyBlYWNoLCB+e3JhdGUgKiAobGVuKHRvZG8pIC0gbnVtYmVyKSAvIDYwOi4wZn0gbWluIGxlZnQiCiAgICAgICAgICAgICkKCgpkZWYgcGFja2FnZSgpIC0+IE5vbmU6CiAgICBzdW1tYXJpZXMgPSB7clsic2hhIl06IHJbInN1bW1hcnkiXSBmb3IgciBpbiByZWFkX2pzb25sKFNVTU1BUklFUykgaWYgci5nZXQoInN1bW1hcnkiKX0KICAgIGNvbW1pdHMgPSByZWFkX2pzb25sKENPTU1JVFMpCiAgICBtaXNzaW5nID0gW2NbInNoYSJdWzoxMl0gZm9yIGMgaW4gY29tbWl0cyBpZiBjWyJzaGEiXSBub3QgaW4gc3VtbWFyaWVzXQogICAgaWYgbWlzc2luZzoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYie2xlbihtaXNzaW5nKX0gY29tbWl0cyBoYXZlIG5vIHN1bW1hcnkgeWV0LCBlLmcuIHttaXNzaW5nWzozXX0iKQogICAgcm93cyA9IFt7KipjLCAic3VtbWFyeSI6IHN1bW1hcmllc1tjWyJzaGEiXV19IGZvciBjIGluIGNvbW1pdHNdCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShQQUNLQUdFLCAidyIsIHppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyBhcmNoaXZlOgogICAgICAgIGFyY2hpdmUud3JpdGVzdHIoImNvbW1pdHMuanNvbmwiLCAiXG4iLmpvaW4oanNvbi5kdW1wcyhyKSBmb3IgciBpbiByb3dzKSArICJcbiIpCiAgICAgICAgYXJjaGl2ZS53cml0ZShRVUVSSUVTLCAicXVlcmllcy5qc29ubCIpCiAgICBwcmludChmIndyb3RlIHtQQUNLQUdFfSAoe1BBQ0tBR0Uuc3RhdCgpLnN0X3NpemUgLyAxZTY6LjFmfSBNQiwge2xlbihyb3dzKX0gY29tbWl0cykiKQoKCmRlZiBzY29yZSh2ZWN0b3JzX3BhdGg6IFBhdGgpIC0+IE5vbmU6CiAgICBmcm9tIGxhbmdjaGFpbl9jb3JlLmRvY3VtZW50cyBpbXBvcnQgRG9jdW1lbnQKCiAgICBmcm9tIHJlcG9fcmFnLmNvbmZpZyBpbXBvcnQgU2V0dGluZ3MKICAgIGZyb20gcmVwb19yYWcuc3RvcmUuYm0yNSBpbXBvcnQgQm0yNUluZGV4CgogICAgZGF0YSA9IG5wLmxvYWQodmVjdG9yc19wYXRoLCBhbGxvd19waWNrbGU9RmFsc2UpCiAgICBzaGFzID0gW3N0cihzKSBmb3IgcyBpbiBkYXRhWyJzaGFzIl1dCiAgICBxdWVyaWVzID0gcmVhZF9qc29ubChRVUVSSUVTKQogICAgY29tbWl0cyA9IHtjWyJzaGEiXTogY1sidGV4dCJdIGZvciBjIGluIHJlYWRfanNvbmwoQ09NTUlUUyl9CiAgICBzdW1tYXJpZXMgPSB7clsic2hhIl06IHJbInN1bW1hcnkiXSBmb3IgciBpbiByZWFkX2pzb25sKFNVTU1BUklFUykgaWYgci5nZXQoInN1bW1hcnkiKX0KICAgIHNldHRpbmdzID0gU2V0dGluZ3MoKQogICAgd2VpZ2h0cyA9IHsiZGVuc2UiOiBzZXR0aW5ncy5kZW5zZV93ZWlnaHQsICJibTI1Ijogc2V0dGluZ3MuYm0yNV93ZWlnaHR9CiAgICB0ZXh0cyA9IHsKICAgICAgICAiYmFzZWxpbmUiOiBbY29tbWl0c1tzXSBmb3IgcyBpbiBzaGFzXSwKICAgICAgICAic3VtbWFyeSI6IFt3aXRoX3N1bW1hcnkoY29tbWl0c1tzXSwgc3VtbWFyaWVzLmdldChzLCAiIikpIGZvciBzIGluIHNoYXNdLAogICAgfQogICAgcmVzdWx0czogZGljdFtzdHIsIEFueV0gPSB7fQogICAgcGVyX3BhaXI6IGRpY3Rbc3RyLCBkaWN0W3N0ciwgbGlzdFtpbnQgfCBOb25lXV1dID0ge30KICAgIGZvciB2YXJpYW50IGluICgiYmFzZWxpbmUiLCAic3VtbWFyeSIpOgogICAgICAgIGRvY3MgPSBbCiAgICAgICAgICAgIERvY3VtZW50KHBhZ2VfY29udGVudD10LCBtZXRhZGF0YT17ImNodW5rX2lkIjogc30pCiAgICAgICAgICAgIGZvciB0LCBzIGluIHppcCh0ZXh0c1t2YXJpYW50XSwgc2hhcywgc3RyaWN0PVRydWUpCiAgICAgICAgXQogICAgICAgIGluZGV4ID0gQm0yNUluZGV4LmJ1aWxkKGRvY3MpCiAgICAgICAgdmVjdG9ycyA9IGRhdGFbZiJ2ZWN0b3JzX3t2YXJpYW50fSJdCiAgICAgICAgZGVuc2VfcmFua3M6IGxpc3RbaW50IHwgTm9uZV0gPSBbXQogICAgICAgIGh5YnJpZF9yYW5rczogbGlzdFtpbnQgfCBOb25lXSA9IFtdCiAgICAgICAgZm9yIHEsIHF2IGluIHppcChxdWVyaWVzLCBkYXRhWyJxdWVyaWVzIl0sIHN0cmljdD1UcnVlKToKICAgICAgICAgICAgb3JkZXIgPSBkZW5zZV9vcmRlcih2ZWN0b3JzLCBxdikKICAgICAgICAgICAgZGVuc2VfcmFua3MuYXBwZW5kKHJhbmtzX2ZvcihvcmRlciwgc2hhcywgcVsiZ29sZCJdKSkKICAgICAgICAgICAgYm0yNSA9IFtjaWQgZm9yIGNpZCwgXyBpbiBpbmRleC5zZWFyY2gocVsicXVlcnkiXSwgbGVuKHNoYXMpKV0KICAgICAgICAgICAgZnVzZWQgPSBoeWJyaWRfb3JkZXIob3JkZXIsIHNoYXMsIGJtMjUsIHdlaWdodHMsIHNldHRpbmdzLnJyZl9rKQogICAgICAgICAgICBoeWJyaWRfcmFua3MuYXBwZW5kKHJhbmtzX2ZvcihmdXNlZCwgc2hhcywgcVsiZ29sZCJdKSkKICAgICAgICBwZXJfcGFpclt2YXJpYW50XSA9IHsiZGVuc2UiOiBkZW5zZV9yYW5rcywgImh5YnJpZCI6IGh5YnJpZF9yYW5rc30KICAgICAgICByZXN1bHRzW3ZhcmlhbnRdID0gewogICAgICAgICAgICAiZGVuc2UiOiBzdW1tYXJpemVfcmFua3MoZGVuc2VfcmFua3MpLAogICAgICAgICAgICAiaHlicmlkIjogc3VtbWFyaXplX3JhbmtzKGh5YnJpZF9yYW5rcyksCiAgICAgICAgfQogICAgZm9yIG1vZGUgaW4gKCJkZW5zZSIsICJoeWJyaWQiKToKICAgICAgICByZXN1bHRzW2YiY2hhbmdlX3ttb2RlfSJdID0gY29tcGFyZV9yYW5rcygKICAgICAgICAgICAgcGVyX3BhaXJbImJhc2VsaW5lIl1bbW9kZV0sIHBlcl9wYWlyWyJzdW1tYXJ5Il1bbW9kZV0KICAgICAgICApCiAgICByZXN1bHRzWyJjb3JwdXMiXSA9IGxlbihzaGFzKQogICAgcmVzdWx0c1sicGFpcnMiXSA9IGxlbihxdWVyaWVzKQogICAgcmVzdWx0c1sic3VtbWFyaWVzIl0gPSBsZW4oc3VtbWFyaWVzKQogICAgUkVTVUxULndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgeyJzdW1tYXJ5IjogcmVzdWx0cywgInJhbmtzIjogcGVyX3BhaXIsICJpZHMiOiBbcVsiaWQiXSBmb3IgcSBpbiBxdWVyaWVzXX0sIGluZGVudD0yCiAgICAgICAgKSwKICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgKQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXN1bHRzLCBpbmRlbnQ9MikpCiAgICBwcmludChmIndyb3RlIHtSRVNVTFR9IikKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgc3ViID0gcGFyc2VyLmFkZF9zdWJwYXJzZXJzKGRlc3Q9ImNvbW1hbmQiLCByZXF1aXJlZD1UcnVlKQogICAgcCA9IHN1Yi5hZGRfcGFyc2VyKCJwcmVwYXJlIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXJhbmRvbSIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMDAsIGhlbHA9InJhbmRvbSBub24tZ29sZCBjb21taXRzIHRvIGFkZCIpCiAgICBzID0gc3ViLmFkZF9wYXJzZXIoInN1bW1hcml6ZSIpCiAgICBzLmFkZF9hcmd1bWVudCgiLS11cmwiLCByZXF1aXJlZD1UcnVlKQogICAgcy5hZGRfYXJndW1lbnQoIi0tdG9rZW4iLCByZXF1aXJlZD1UcnVlKQogICAgcy5hZGRfYXJndW1lbnQoIi0tbGltaXQiLCB0eXBlPWludCwgaGVscD0ic3VtbWFyaXNlIGF0IG1vc3QgdGhpcyBtYW55IG1vcmUgKHRvIHRyeSBpdCBvdXQpIikKICAgIHN1Yi5hZGRfcGFyc2VyKCJwYWNrYWdlIikKICAgIGMgPSBzdWIuYWRkX3BhcnNlcigic2NvcmUiKQogICAgYy5hZGRfYXJndW1lbnQoIi0tdmVjdG9ycyIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBpZiBhcmdzLmNvbW1hbmQgPT0gInByZXBhcmUiOgogICAgICAgIHByZXBhcmUoYXJncy5yYW5kb20pCiAgICBlbGlmIGFyZ3MuY29tbWFuZCA9PSAic3VtbWFyaXplIjoKICAgICAgICBzdW1tYXJpemUoYXJncy51cmwsIGFyZ3MudG9rZW4sIGFyZ3MubGltaXQpCiAgICBlbGlmIGFyZ3MuY29tbWFuZCA9PSAicGFja2FnZSI6CiAgICAgICAgcGFja2FnZSgpCiAgICBlbHNlOgogICAgICAgIHNjb3JlKGFyZ3MudmVjdG9ycykKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="))

In [ ]:
# Optional: a Hugging Face token makes the model download faster and avoids rate limits.
import os

try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
    print("HF_TOKEN:", "set" if token else "not set")
except Exception as exc:  # no secret, or not running in Colab
    print("HF_TOKEN: not set (", type(exc).__name__, ") - continuing without it")

## Parameters

In [ ]:
# --- Parameters -------------------------------------------------------------
DTYPE = "float32"      # matches the laptop's queries; "bfloat16" is faster and nearly identical
DOC_BATCH = 8          # commit texts run to 2,048 tokens: large batches ran out of 40 GB before

## Upload the pilot data

In [ ]:
import io, json, os, zipfile

from google.colab import files

uploaded = files.upload()   # pilot_input.zip from `commit_summary_pilot.py package`
assert len(uploaded) == 1, "Upload exactly one file: pilot_input.zip"
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    commits = [json.loads(x) for x in archive.read("commits.jsonl").decode().splitlines() if x]
    queries = [json.loads(x) for x in archive.read("queries.jsonl").decode().splitlines() if x]
assert all(c.get("summary") for c in commits), "some commits have no summary: rerun summarize"
print(f"{len(commits)} commits, {len(queries)} queries")
print("example summary:", commits[0]["summary"][:300])

## Embed with and without the summary

In [ ]:
import os, sys, time

os.environ["PYTHONPATH"] = str(SRC)
os.environ["REPO_RAG_EMBEDDING_DTYPE"] = DTYPE
os.environ["REPO_RAG_EMBEDDING_BATCH_SIZE"] = str(DOC_BATCH)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path[:0] = [str(SRC), "/content"]

import numpy as np
import torch

from repo_rag.config import Settings
from repo_rag.store.embeddings import LocalEmbeddings

import commit_summary_pilot as pilot

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (this will take hours)"
print("GPU:", gpu)
embedder = LocalEmbeddings(Settings())
shas = [c["sha"] for c in commits]
texts = {
    "baseline": [c["text"] for c in commits],
    "summary": [pilot.with_summary(c["text"], c["summary"]) for c in commits],
}
vectors = {}
for name, batch in texts.items():
    started = time.perf_counter()
    vectors[name] = np.asarray(embedder.embed_documents(batch), dtype=np.float32)
    print(f"{name}: {vectors[name].shape} in {time.perf_counter() - started:.0f} s")
started = time.perf_counter()
query_vectors = np.asarray([embedder.embed_query(q["query"]) for q in queries], dtype=np.float32)
print(f"queries: {query_vectors.shape} in {time.perf_counter() - started:.0f} s")

In [ ]:
# A first answer on the GPU box: dense search only, the gold commit's rank inside this corpus.
ranks = {}
for name in ("baseline", "summary"):
    ranks[name] = [
        pilot.ranks_for(pilot.dense_order(vectors[name], qv), shas, q["gold"])
        for q, qv in zip(queries, query_vectors)
    ]
    print(f"{name:9s}", pilot.summarize_ranks(ranks[name]))
print("summary vs baseline:", pilot.compare_ranks(ranks["baseline"], ranks["summary"]))

## Download

In [ ]:
np.savez_compressed(
    "/content/pilot_vectors.npz",
    shas=np.array(shas),
    queries=query_vectors,
    vectors_baseline=vectors["baseline"],
    vectors_summary=vectors["summary"],
)
print(f"pilot_vectors.npz: {os.path.getsize('/content/pilot_vectors.npz') / 1e6:.1f} MB")

from google.colab import files

files.download("/content/pilot_vectors.npz")

## On your machine

```bash
uv run python eval/commit_summary_pilot.py score --vectors path\to\pilot_vectors.npz
```

It prints hit@1/5/10/20 and MRR with and without the summary, for dense search and for the
hybrid search the agent uses, and how many of the gold commits moved up or down. Colab wipes
`/content` when the session ends, so download the vectors before closing the tab.